# 🔤 Notebook 03: Tokenization

LLMs don't read text — they read numbers. **Tokenization** is the bridge between human language and the integer sequences that transformers actually process. Every prompt you type, every response you read, passes through a tokenizer on the way in and on the way out.

This notebook builds tokenizers from scratch, starting with the simplest possible approach and working up to the production tokenizers used by GPT-4 and LLaMA. By the end, you'll understand exactly how `"Hello, world!"` becomes `[9906, 11, 1917, 0]` — and why that mapping matters for model performance.

**What we'll cover:**
1. **Character-level tokenization** — the simplest baseline (and why it's too slow)
2. **Byte-Pair Encoding (BPE) from scratch** — the algorithm behind GPT-2/3/4
3. **tiktoken** — OpenAI's production tokenizer for GPT-4
4. **SentencePiece** — Google's tokenizer used in LLaMA and Gemma

**Prerequisites:** Notebooks 00–02 (environment, MLX basics, math foundations)

## ✅ Environment Validation

Let's confirm our environment is ready before we start tokenizing.

In [1]:
from utils.checks import validate_environment, print_environment_report

results = print_environment_report()

# Check tokenization-specific dependencies
import importlib

tok_deps = {"tiktoken": "tiktoken", "sentencepiece": "sentencepiece"}
print("\n📦 Tokenization dependencies:")
for name, pkg in tok_deps.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", getattr(mod, "VERSION", "installed"))
        print(f"  ✅ {name}: {ver}")
    except ImportError:
        print(f"  ❌ {name}: not installed — run: pip install {pkg}")

  Environment Validation Report
  Platform : macOS-26.4-arm64-arm-64bit-Mach-O
  Chip     : Apple M4 Pro
  Python   : 3.13.13  ✅
  MLX      : 0.31.1  ✅
  Metal GPU: available  ✅
  Memory   : 48.0 GB  ✅

📦 Tokenization dependencies:
  ✅ tiktoken: 0.12.0
  ✅ sentencepiece: 0.2.1


---
## 📜 The Evolution of Tokenization

Tokenization has evolved through four major stages, each solving problems the previous one couldn't:

| Era | Method | Vocabulary | Token for "unhappiness" | Problem |
|-----|--------|-----------|------------------------|---------|
| 1️⃣ | **Character-level** | ~100 chars | `['u','n','h','a','p','p','i','n','e','s','s']` (11 tokens) | Sequences too long, no word meaning |
| 2️⃣ | **Word-level** | 100K+ words | `['unhappiness']` (1 token) | Huge vocab, can't handle new words (OOV) |
| 3️⃣ | **BPE (Byte-Pair Encoding)** | 30K–100K subwords | `['un', 'happiness']` (2 tokens) | Sweet spot — compact and flexible |
| 4️⃣ | **Byte-level BPE** | 256 bytes + merges | `['un', 'happ', 'iness']` (3 tokens) | Handles ANY text (code, emoji, multilingual) |

💡 **Insight:** Modern LLMs (GPT-4, LLaMA, Gemma) all use variants of BPE. The key idea: **start with individual bytes/characters, then iteratively merge the most frequent pairs** until you reach your target vocabulary size.

🎯 **Interview tip:** "Why not just use words as tokens?" — Because (1) the vocabulary would be enormous, (2) you can't handle misspellings or new words, and (3) morphologically rich languages like Turkish or Finnish would need millions of entries. Subword tokenization solves all three.

---
## 🔡 Character-Level Tokenization

The simplest tokenizer: every unique character gets its own integer ID. We'll build one from scratch to understand the core encode/decode contract that ALL tokenizers must satisfy:

$$\text{decode}(\text{encode}(s)) = s \quad \forall s$$

This **lossless roundtrip** property is non-negotiable — if your tokenizer loses information, your model can never recover it.

### Building a Character Vocabulary

First, we extract every unique character from our text and sort them to create a deterministic mapping. The vocabulary size equals the number of unique characters.

In [2]:
# Sample text — a small corpus to tokenize
text = """The transformer architecture revolutionized natural language processing.
Attention is all you need. Self-attention lets every token look at every other token.
GPT, LLaMA, and Gemma all use byte-pair encoding for tokenization."""

# Build vocabulary: sorted unique characters
chars = sorted(set(text))
vocab_size = len(chars)

# Create char → int and int → char mappings
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for ch, i in char_to_id.items()}

print(f"Text length: {len(text)} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"\nVocabulary: {''.join(chars)}")
print(f"\nSample mappings:")
for ch in ['A', 'a', ' ', '.', 'T']:
    if ch in char_to_id:
        print(f"  '{ch}' → {char_to_id[ch]}")

Text length: 225 characters
Vocabulary size: 34 unique characters

Vocabulary: 
 ,-.AGLMPSTabcdefghiklmnoprstuvyz

Sample mappings:
  'A' → 5
  'a' → 12
  ' ' → 1
  '.' → 4
  'T' → 11


### Encode and Decode Functions

The two essential operations: `encode` converts a string to a list of integers, `decode` converts back. Together they must be lossless.

In [3]:
def char_encode(s: str, char_to_id: dict) -> list[int]:
    """Encode a string into a list of character-level token IDs."""
    return [char_to_id[ch] for ch in s]

def char_decode(ids: list[int], id_to_char: dict) -> str:
    """Decode a list of token IDs back into a string."""
    return "".join(id_to_char[i] for i in ids)

# Encode a sample
sample = "Attention is all you need."
encoded = char_encode(sample, char_to_id)
decoded = char_decode(encoded, id_to_char)

print(f"Original:  '{sample}'")
print(f"Encoded:   {encoded}")
print(f"Decoded:   '{decoded}'")
print(f"Roundtrip: {'✅ PASS' if decoded == sample else '❌ FAIL'}")
print(f"\nToken count: {len(encoded)} tokens for {len(sample)} characters")
print(f"Ratio: {len(encoded) / len(sample):.2f} tokens per character (always 1.0 for char-level)")

Original:  'Attention is all you need.'
Encoded:   [5, 29, 29, 16, 24, 29, 20, 25, 24, 1, 20, 28, 1, 12, 22, 22, 1, 32, 25, 30, 1, 24, 16, 16, 15, 4]
Decoded:   'Attention is all you need.'
Roundtrip: ✅ PASS

Token count: 26 tokens for 26 characters
Ratio: 1.00 tokens per character (always 1.0 for char-level)


### ⚠️ The Inefficiency Problem: Character-Level vs Subword

Character-level tokenization has a fatal flaw: **1 character = 1 token**. This means sequences are extremely long, and the model must learn to compose characters into words from scratch. Let's see how this compares to BPE.

In [ ]:
import tiktoken

# Load GPT-4's tokenizer for comparison
enc = tiktoken.get_encoding("cl100k_base")

test_texts = [
    "The transformer architecture revolutionized NLP.",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "Attention is all you need.",
]

print("=" * 70)
print(f"{'Text':<45} {'Char Tokens':>12} {'BPE Tokens':>11} {'Ratio':>7}")
print("=" * 70)

for t in test_texts:
    char_count = len(t)  # 1 char = 1 token
    bpe_count = len(enc.encode(t))
    ratio = char_count / bpe_count
    short = t[:42] + "..." if len(t) > 42 else t
    print(f"{short:<45} {char_count:>12} {bpe_count:>11} {ratio:>6.1f}x")

print("=" * 70)
print("\n💡 BPE uses 3-5x fewer tokens than character-level.")
print("   Fewer tokens → shorter sequences → faster training → lower cost.")

---
## 🔧 Byte-Pair Encoding (BPE) from Scratch

BPE is the algorithm behind GPT-2, GPT-3, GPT-4, and many other LLMs. The idea is beautifully simple:

1. Start with individual bytes (256 possible tokens)
2. Count all adjacent pairs in the corpus
3. Merge the most frequent pair into a new token
4. Repeat until you reach your target vocabulary size

Each merge reduces the total token count, compressing the representation. Let's implement it step by step.

### Implementing `train_bpe()` — The Core Algorithm

We start with raw bytes and iteratively merge the most frequent adjacent pair. At each step we print what's happening so you can see the algorithm in action.

In [ ]:
def get_pair_counts(token_ids: list[int]) -> dict[tuple[int, int], int]:
    """Count all adjacent pairs in the token list."""
    counts = {}
    for i in range(len(token_ids) - 1):
        pair = (token_ids[i], token_ids[i + 1])
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge_pair(token_ids: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    """Replace all occurrences of `pair` in token_ids with `new_id`."""
    merged = []
    i = 0
    while i < len(token_ids):
        if i < len(token_ids) - 1 and (token_ids[i], token_ids[i + 1]) == pair:
            merged.append(new_id)
            i += 2
        else:
            merged.append(token_ids[i])
            i += 1
    return merged


def train_bpe(corpus: str, num_merges: int, verbose: bool = True) -> tuple[dict, list, list]:
    """Train a BPE tokenizer from scratch.
    
    Args:
        corpus: raw text to learn vocabulary from
        num_merges: number of merge operations to perform
        verbose: print each merge step
    
    Returns:
        vocab: dict mapping token bytes/tuples to integer IDs
        merges: list of (pair, new_id) merge operations in order
        tokens: final token list after all merges applied to corpus
    """
    # Step 1: Start with raw bytes
    tokens = list(corpus.encode("utf-8"))
    initial_len = len(tokens)
    
    # Base vocabulary: 256 byte values
    vocab = {bytes([i]): i for i in range(256)}
    merges = []
    next_id = 256
    
    if verbose:
        print(f"Starting: {initial_len} tokens (raw bytes)")
        print(f"Target: {num_merges} merges → vocab size {256 + num_merges}")
        print("-" * 60)
    
    for step in range(num_merges):
        # Count adjacent pairs
        pair_counts = get_pair_counts(tokens)
        if not pair_counts:
            if verbose:
                print(f"  No more pairs to merge at step {step}.")
            break
        
        # Find the most frequent pair
        best_pair = max(pair_counts, key=pair_counts.get)
        best_count = pair_counts[best_pair]
        
        # Merge it
        tokens = merge_pair(tokens, best_pair, next_id)
        
        # Record the merge
        merges.append((best_pair, next_id))
        vocab[best_pair] = next_id
        
        if verbose:
            # Decode pair bytes for display
            def id_to_str(tid):
                if tid < 256:
                    return repr(chr(tid)) if 32 <= tid < 127 else f"0x{tid:02x}"
                return f"tok_{tid}"
            
            p0, p1 = id_to_str(best_pair[0]), id_to_str(best_pair[1])
            print(f"  Merge {step + 1:>3}: ({p0}, {p1}) → tok_{next_id}"
                  f"  (count: {best_count}, tokens: {initial_len} → {len(tokens)})")
        
        next_id += 1
    
    if verbose:
        print("-" * 60)
        print(f"Done! Vocab size: {next_id}, Tokens: {initial_len} → {len(tokens)}")
    
    return vocab, merges, tokens


# Train on our sample text with 20 merges
corpus = """The transformer architecture revolutionized natural language processing.
Attention is all you need. Self-attention lets every token look at every other token.
GPT, LLaMA, and Gemma all use byte-pair encoding for tokenization."""

vocab, merges, final_tokens = train_bpe(corpus, num_merges=20)

### ⚡ Compression Ratio

The whole point of BPE is compression: represent the same text with fewer tokens. Let's measure how well our tiny BPE tokenizer compresses the corpus.

In [ ]:
# Compression analysis
raw_bytes = list(corpus.encode("utf-8"))
raw_len = len(raw_bytes)
bpe_len = len(final_tokens)
compression_ratio = raw_len / bpe_len

print(f"Raw byte tokens:  {raw_len}")
print(f"After BPE tokens: {bpe_len}")
print(f"Compression ratio: {compression_ratio:.2f}x")
print(f"Space saved: {(1 - bpe_len / raw_len) * 100:.1f}%")
print(f"\n💡 With only 20 merges we already achieved {compression_ratio:.2f}x compression.")
print(f"   GPT-4 uses ~100K merges for much higher compression on diverse text.")

### Encode and Decode with Learned Merges

Now let's build encode/decode functions that use our learned BPE merges. Encoding applies merges in the same order they were learned. Decoding maps token IDs back to bytes.

In [ ]:
def bpe_encode(text: str, merges: list[tuple[tuple[int, int], int]]) -> list[int]:
    """Encode text using learned BPE merges.
    
    Start with raw bytes, then apply each merge rule in order.
    """
    tokens = list(text.encode("utf-8"))
    for pair, new_id in merges:
        tokens = merge_pair(tokens, pair, new_id)
    return tokens


def bpe_decode(token_ids: list[int], merges: list[tuple[tuple[int, int], int]]) -> str:
    """Decode BPE token IDs back to text.
    
    Reverse each merge to recover the original bytes, then decode UTF-8.
    """
    # Build reverse mapping: new_id → (left, right)
    id_to_pair = {new_id: pair for pair, new_id in merges}
    
    def expand(tid: int) -> list[int]:
        """Recursively expand a token ID to its base bytes."""
        if tid < 256:
            return [tid]
        if tid in id_to_pair:
            left, right = id_to_pair[tid]
            return expand(left) + expand(right)
        return [tid]
    
    # Expand all tokens to bytes
    byte_list = []
    for tid in token_ids:
        byte_list.extend(expand(tid))
    
    return bytes(byte_list).decode("utf-8")


# Test roundtrip
test_str = "Attention is all you need."
encoded = bpe_encode(test_str, merges)
decoded = bpe_decode(encoded, merges)

print(f"Original:  '{test_str}'")
print(f"Encoded:   {encoded}")
print(f"# tokens:  {len(encoded)} (vs {len(test_str)} chars)")
print(f"Decoded:   '{decoded}'")
print(f"Roundtrip: {'✅ PASS' if decoded == test_str else '❌ FAIL'}")

# Test on the full corpus
full_encoded = bpe_encode(corpus, merges)
full_decoded = bpe_decode(full_encoded, merges)
print(f"\nFull corpus roundtrip: {'✅ PASS' if full_decoded == corpus else '❌ FAIL'}")

🎯 **Interview tip:** "Walk me through how BPE works." — Start with bytes (256 tokens), count adjacent pairs, merge the most frequent one into a new token, repeat N times. The merge list IS the tokenizer — encoding applies merges in order, decoding reverses them. The key insight: frequent subwords get their own tokens, rare words decompose into smaller pieces.

⚠️ **Pitfall:** BPE merge order matters! Applying merges in a different order can produce different tokenizations. Production tokenizers store the exact merge list and apply them deterministically.

---
## 🏭 tiktoken: GPT-4's Production Tokenizer

OpenAI's `tiktoken` is a fast, production-grade BPE tokenizer. GPT-4 uses the `cl100k_base` encoding with ~100,000 tokens. Let's explore how it tokenizes different types of text.

### Loading the GPT-4 Tokenizer

`cl100k_base` is the encoding used by GPT-4, GPT-3.5-turbo, and text-embedding-ada-002. It has 100,256 tokens and handles English, code, and multilingual text well.

In [ ]:
import tiktoken

# Load GPT-4's tokenizer
enc = tiktoken.get_encoding("cl100k_base")

print(f"Encoding name: {enc.name}")
print(f"Vocabulary size: {enc.n_vocab:,} tokens")
print(f"Max token value: {enc.max_token_value:,}")

### Encoding Various Text Types

Let's see how tiktoken handles English prose, Python code, and multilingual text. Notice how common English words often get a single token, while rare words and code get split into subwords.

In [ ]:
texts = {
    "English": "The quick brown fox jumps over the lazy dog.",
    "Code": "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "Multilingual": "こんにちは世界! Bonjour le monde! مرحبا بالعالم",
}

for label, text in texts.items():
    token_ids = enc.encode(text)
    # Decode each token individually to see the pieces
    decoded_tokens = [enc.decode([tid]) for tid in token_ids]
    
    print(f"\n{'=' * 60}")
    print(f"📝 {label}: \"{text}\"")
    print(f"   Token IDs ({len(token_ids)} tokens): {token_ids}")
    print(f"   Decoded tokens: {decoded_tokens}")

### Token IDs and Decoded Tokens — A Closer Look

Let's visualize the token boundaries for a sentence, showing exactly where tiktoken splits the text and what ID each piece gets.

In [ ]:
sample = "Tokenization is surprisingly important for LLM performance."
token_ids = enc.encode(sample)

print(f"Text: \"{sample}\"")
print(f"Total tokens: {len(token_ids)}\n")
print(f"{'Index':<7} {'Token ID':<10} {'Decoded':>20}")
print("-" * 40)
for i, tid in enumerate(token_ids):
    decoded = enc.decode([tid])
    print(f"{i:<7} {tid:<10} {repr(decoded):>20}")

### Fertility: Tokens per Word

**Fertility** measures how many tokens a tokenizer uses per word. Lower fertility = more efficient. English typically gets ~1.3 tokens/word with GPT-4's tokenizer, while other languages can be 2-4x higher — this is a known equity issue in LLM pricing and context windows.

In [ ]:
fertility_texts = {
    "English": "The quick brown fox jumps over the lazy dog",
    "Python code": "def hello(): print('Hello, world!')",
    "Japanese": "東京は日本の首都です",
    "French": "Bonjour le monde comment allez vous",
    "Arabic": "مرحبا بالعالم كيف حالك",
    "Technical": "The transformer uses multi-head self-attention with RMSNorm",
}

print(f"{'Language':<18} {'Words':>6} {'Tokens':>7} {'Fertility':>10}")
print("=" * 45)

for label, text in fertility_texts.items():
    words = text.split()
    n_words = len(words)
    n_tokens = len(enc.encode(text))
    fertility = n_tokens / n_words if n_words > 0 else 0
    print(f"{label:<18} {n_words:>6} {n_tokens:>7} {fertility:>9.2f}x")

print("\n💡 English and French are efficient (~1.3 tok/word).")
print("   CJK and Arabic use more tokens per word — each character")
print("   may need multiple byte-level tokens.")

⚡ **Performance note:** tiktoken is implemented in Rust and is extremely fast — it can tokenize millions of tokens per second. Our from-scratch BPE is ~100x slower but teaches the same algorithm.

🎯 **Interview tip:** "Why does GPT-4 use ~100K tokens?" — It's a tradeoff. Larger vocab = shorter sequences (faster inference, more context) but larger embedding table (more parameters). Smaller vocab = longer sequences but smaller model. ~100K is the sweet spot found empirically.

---
## 📦 SentencePiece: Google's Tokenizer

SentencePiece (used by LLaMA, Gemma, T5, and many others) takes a different approach from tiktoken. It treats the input as a raw byte stream — **no pre-tokenization** (no splitting on spaces first). This makes it truly language-agnostic.

SentencePiece supports two algorithms:

| Algorithm | How it works | Used by |
|-----------|-------------|---------|
| **BPE** | Bottom-up: merge frequent pairs (same idea as above) | LLaMA, Mistral |
| **Unigram** | Top-down: start with a large vocab, prune least useful tokens | T5, ALBERT, XLNet |

💡 **Insight:** The **unigram model** is the opposite of BPE. Instead of building up from characters, it starts with a huge candidate vocabulary and iteratively removes tokens that contribute least to the corpus likelihood. It uses a probabilistic model: each token has a probability, and the best tokenization of a string maximizes the product of token probabilities.

### Demonstrating SentencePiece with a Pretrained Model

We'll load a pretrained SentencePiece model from a HuggingFace model (LLaMA-style) to see how it tokenizes text. If the model isn't available, we'll train a small one locally to demonstrate the concept.

In [ ]:
import sentencepiece as spm
import tempfile
import os

# Train a small SentencePiece BPE model on our corpus
# This demonstrates the full pipeline: text → model → encode/decode

training_text = """The transformer architecture revolutionized natural language processing.
Attention is all you need. Self-attention lets every token look at every other token.
GPT LLaMA and Gemma all use byte-pair encoding for tokenization.
Large language models process text as sequences of integer token IDs.
Tokenization is the bridge between human language and neural networks.
The vocabulary size affects model size and sequence length tradeoffs.
Subword tokenization handles rare words by splitting them into pieces.
Byte-pair encoding starts with characters and merges frequent pairs.
SentencePiece treats input as a raw byte stream without pre-tokenization.
The unigram model starts large and prunes the least useful tokens."""

# Write training text to a temp file (SentencePiece requires a file)
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write(training_text)
    train_file = f.name

# Train a small SentencePiece BPE model
model_prefix = os.path.join(tempfile.gettempdir(), "sp_demo")
spm.SentencePieceTrainer.train(
    input=train_file,
    model_prefix=model_prefix,
    vocab_size=150,           # Small vocab for demonstration
    model_type="bpe",         # BPE algorithm (like LLaMA)
    character_coverage=1.0,   # Cover all characters in training data
    pad_id=3,                 # Padding token ID
)

# Load the trained model
sp = spm.SentencePieceProcessor()
sp.load(f"{model_prefix}.model")

print(f"✅ SentencePiece model trained!")
print(f"   Vocab size: {sp.get_piece_size()}")
print(f"   Model type: BPE")

# Clean up temp file
os.unlink(train_file)

### SentencePiece Encode/Decode

Let's see how our trained SentencePiece model tokenizes text. Notice the `▁` (U+2581) character — SentencePiece uses this to mark word boundaries since it doesn't pre-split on spaces.

In [ ]:
test = "Attention is all you need."

# Encode to token IDs
sp_ids = sp.encode(test, out_type=int)
# Encode to token pieces (strings)
sp_pieces = sp.encode(test, out_type=str)
# Decode back
sp_decoded = sp.decode(sp_ids)

print(f"Text:       \"{test}\"")
print(f"Token IDs:  {sp_ids}")
print(f"Pieces:     {sp_pieces}")
print(f"Decoded:    \"{sp_decoded}\"")
print(f"Roundtrip:  {'✅ PASS' if sp_decoded == test else '❌ FAIL'}")
print(f"\n💡 The ▁ character marks where spaces were in the original text.")

### 🏁 Grand Comparison: All Four Tokenizers on the Same Text

Let's tokenize the same sentence with all four methods and compare token counts, pieces, and efficiency. This is the key takeaway of this notebook.

In [ ]:
import matplotlib.pyplot as plt

comparison_text = "Attention is all you need."

# 1. Character-level
char_tokens = list(comparison_text)
char_count = len(char_tokens)

# 2. Our BPE from scratch
bpe_ids = bpe_encode(comparison_text, merges)
bpe_count = len(bpe_ids)

# 3. tiktoken (GPT-4)
tiktoken_ids = enc.encode(comparison_text)
tiktoken_pieces = [enc.decode([tid]) for tid in tiktoken_ids]
tiktoken_count = len(tiktoken_ids)

# 4. SentencePiece
sp_pieces_list = sp.encode(comparison_text, out_type=str)
sp_count = len(sp_pieces_list)

# Print comparison
print(f"Text: \"{comparison_text}\"\n")
print(f"{'Tokenizer':<20} {'# Tokens':>9}  Pieces")
print("=" * 70)
print(f"{'Character':<20} {char_count:>9}  {char_tokens}")
print(f"{'Our BPE (20 merges)':<20} {bpe_count:>9}  (token IDs: {bpe_ids})")
print(f"{'tiktoken (cl100k)':<20} {tiktoken_count:>9}  {tiktoken_pieces}")
print(f"{'SentencePiece':<20} {sp_count:>9}  {sp_pieces_list}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
methods = ["Character", "Our BPE\n(20 merges)", "tiktoken\n(GPT-4)", "SentencePiece\n(150 vocab)"]
counts = [char_count, bpe_count, tiktoken_count, sp_count]
colors = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]

bars = ax.bar(methods, counts, color=colors, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Number of Tokens")
ax.set_title(f'Token Count Comparison: "{comparison_text}"')

# Add count labels on bars
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(count), ha="center", va="bottom", fontweight="bold")

ax.set_ylim(0, max(counts) + 4)
plt.tight_layout()
plt.show()

### Extended Comparison Across Multiple Texts

Let's compare all tokenizers on a variety of inputs to see how they handle different content types.

In [ ]:
comparison_texts = {
    "Simple English": "The cat sat on the mat.",
    "Technical": "Self-attention uses scaled dot-product.",
    "Code snippet": "for i in range(10): print(i)",
    "Mixed": "GPT-4 uses cl100k_base tokenizer!",
}

print(f"{'Text':<42} {'Char':>5} {'BPE':>5} {'tiktoken':>9} {'SP':>5}")
print("=" * 72)

for label, text in comparison_texts.items():
    c = len(text)
    b = len(bpe_encode(text, merges))
    t = len(enc.encode(text))
    s = len(sp.encode(text, out_type=int))
    display = f"{label}: {text}"
    if len(display) > 40:
        display = display[:37] + "..."
    print(f"{display:<42} {c:>5} {b:>5} {t:>9} {s:>5}")

print("\n⚡ tiktoken (100K vocab) consistently uses the fewest tokens.")
print("   Our BPE (20 merges) is limited but shows the same principle.")

---
## 🧭 Tokenizer Selection Guide

With so many tokenizer options, how do you pick the right one? Here's a plain-English decision tree:

### Which Tokenizer Should I Use?

Start with this question: **Are you training a model from scratch, or using a pretrained one?**

- **Using a pretrained model?** → Use whatever tokenizer it was trained with. No exceptions. Mismatched tokenizers silently corrupt every embedding lookup.
- **Training from scratch?** → Read on.

### The Four Main Algorithms

Think of tokenizer algorithms like different strategies for breaking text into pieces:

- **BPE (Byte-Pair Encoding)** — Builds vocabulary bottom-up. Starts with individual bytes, then repeatedly merges the most common pair. Like learning to read by first memorizing letters, then common letter combos ("th", "ing"), then whole words.

- **WordPiece** — Very similar to BPE, but picks merges based on which pair *most improves the likelihood of the training data* rather than raw frequency. Think of it as a slightly smarter version of BPE.

- **Unigram** — Works top-down. Starts with a huge vocabulary and removes the least useful tokens one by one. Like starting with a full dictionary and crossing out words you never use.

- **SentencePiece** — Not an algorithm itself, but a *framework* that can run either BPE or Unigram. Its superpower: it treats text as raw bytes, so it works on any language without needing language-specific rules.

### Decision Table

| Your Situation | Recommended Tokenizer | Why |
|---|---|---|
| Building a GPT-style model | **BPE** (via tiktoken or SentencePiece) | Battle-tested, great compression, deterministic |
| Building a BERT-style model | **WordPiece** | Standard for encoder models, Google ecosystem |
| Multilingual model | **SentencePiece (Unigram)** | Language-agnostic, handles scripts without pre-tokenization |
| Code generation model | **Byte-level BPE** | Handles any character (brackets, operators, whitespace) without UNK tokens |
| Resource-constrained / mobile | **BPE with smaller vocab (32K)** | Smaller embedding table, less memory |
| Maximum context efficiency | **BPE with larger vocab (100K+)** | Fewer tokens per text = more fits in context window |
| Research / data augmentation | **Unigram** | Can sample multiple valid tokenizations of the same text |

### 💡 Rule of Thumb

> If you're unsure, use **SentencePiece with BPE** and a vocab size of 32K–64K. This is what LLaMA, Mistral, and most modern open models use. It works well across languages, handles any input, and gives good compression.

### 🎯 Interview Tip

> *"The choice of tokenizer algorithm (BPE vs Unigram) matters less than the choice of vocabulary size and training data. A BPE tokenizer trained on English-only data will be terrible for Japanese regardless of the algorithm. The training corpus composition is the single biggest factor in tokenizer quality."*

---
## 🏁 Summary

You now understand the full tokenization pipeline that every LLM uses:

| What you learned | Why it matters |
|-----------------|---------------|
| **Character-level** tokenization | Simplest baseline — shows why we need subwords |
| **BPE from scratch** | The actual algorithm behind GPT-2/3/4 tokenizers |
| **tiktoken** (production BPE) | How GPT-4 tokenizes text in practice |
| **SentencePiece** | Language-agnostic tokenizer used by LLaMA/Gemma |
| **Fertility analysis** | Why tokenizer choice affects multilingual fairness |

🎯 **Key takeaway for interviews:** Tokenization is not just preprocessing — it fundamentally shapes what the model can learn. A bad tokenizer wastes context window on redundant tokens. A good tokenizer compresses common patterns while gracefully handling rare inputs.

**Next up:** Notebook 04 — Embeddings & Positional Encoding, where we turn these token IDs into dense vectors the transformer can actually compute with.

---
## 🔮 What Happens After Tokenization?

You've turned text into a list of integer IDs. But a neural network can't do much with raw integers — it needs *dense vectors* it can compute with. So what comes next?

### The Pipeline: Text → Tokens → Embeddings → Transformer

```
"Hello, world!" → [9906, 11, 1917, 0] → [[0.12, -0.34, ...], [...], [...], [...]] → Transformer
     text              token IDs              embedding vectors              magic happens
```

Each token ID becomes a lookup into an **embedding table** — a big matrix where row `i` is the learned vector for token `i`. These vectors are what the transformer actually reads and writes.

### 💡 Three Types of Transformer Models

A helpful analogy — think of language tasks like reading, writing, and translating:

| Type | What It Does | Analogy | Examples |
|------|-------------|---------|----------|
| **Encoder** | Reads text and builds understanding | 📖 Reading comprehension — you read a passage and answer questions about it | BERT, RoBERTa |
| **Decoder** | Writes text one token at a time | ✍️ Creative writing — you write the next word based on everything before it | GPT-4, LLaMA, Gemma |
| **Encoder-Decoder** | Reads input, then writes output | 🌐 Translation — you read in one language and write in another | T5, BART |

Most modern LLMs (GPT-4, LLaMA, Gemma) are **decoder-only** models. They predict the next token given all previous tokens — that's it. The magic is that this simple objective, scaled up, produces remarkably capable systems.

### 🎯 The Journey of a Token

Here's the full path a token takes through an LLM:

1. **Tokenization** (this notebook) — text becomes integer IDs
2. **Embedding lookup** — each ID becomes a dense vector (e.g., 4096 dimensions)
3. **Positional encoding** — position information is added so the model knows word order
4. **Transformer layers** — self-attention and feed-forward networks transform the vectors
5. **Output projection** — the final vector is projected back to vocabulary size
6. **Sampling** — a token ID is chosen from the probability distribution
7. **Detokenization** — the ID is converted back to text

Steps 2 and 3 are exactly what we'll build in the next notebook.

### ➡️ Next Up: Notebook 04 — Embeddings & Positional Encoding

Now that you understand how text becomes token IDs, the next notebook answers: *How do we turn those IDs into vectors the transformer can compute with? And how does the model know that "cat" in position 1 is different from "cat" in position 50?*

We'll implement embedding tables from scratch, explore sinusoidal and RoPE positional encodings, and see why position information is essential for transformers (hint: without it, "the cat sat on the mat" and "mat the on sat cat the" look identical to the model).

## 📜 History & Alternatives

### The Evolution of Text Tokenization

Tokenization is the critical bridge between human text and model computation. The field evolved from simple word splitting to sophisticated subword algorithms that balance vocabulary size, coverage, and compression efficiency.

| Year | Method | Team | Key Contribution |
|------|--------|------|-----------------|
| Pre-2010 | **Word-level** | Traditional NLP | One token per word — simple but huge vocabularies, can't handle OOV words |
| Pre-2010 | **Character-level** | Various | One token per character — tiny vocab but very long sequences |
| 2016 | **BPE (Byte Pair Encoding)** | Sennrich et al. | Iteratively merge most frequent byte pairs — used in GPT-2, GPT-3, LLaMA |
| 2016 | **WordPiece** | Schuster & Nakajima (Google) | Similar to BPE but uses likelihood-based merging — used in BERT, Gemini |
| 2018 | **Unigram LM** | Kudo (Google) | Probabilistic model, starts large and prunes — used in T5, ALBERT |
| 2018 | **SentencePiece** | Kudo & Richardson (Google) | Language-agnostic tokenizer — treats input as raw bytes, no pre-tokenization needed |
| 2022 | **tiktoken** | OpenAI | Fast BPE implementation in Rust — used in GPT-3.5, GPT-4 |
| 2023 | **Byte-level BPE** | Meta AI (LLaMA) | BPE directly on UTF-8 bytes — no unknown tokens ever, 32K vocab |
| 2024 | **Multi-modal tokenizers** | Various | Unified tokenizers handling text + image + audio tokens |

### 💡 Key Insight: The Tokenization Trilemma

Every tokenizer balances three competing goals:

```
Vocabulary Size ←→ Sequence Length ←→ Coverage
     (small)           (short)         (complete)
```

- **Smaller vocab** → longer sequences (more tokens per text) → slower inference
- **Larger vocab** → shorter sequences → larger embedding tables → more memory
- **BPE/WordPiece** hit the sweet spot: 32K-128K vocab covers most text efficiently

### Tokenizer Comparison

| Tokenizer | Vocab Size | Algorithm | Used By | Strengths |
|-----------|-----------|-----------|---------|-----------|
| BPE (tiktoken) | 100K | Byte-pair merging | GPT-4, GPT-4o | Fast, good compression |
| SentencePiece (BPE) | 32K | Byte-pair on raw bytes | LLaMA 1/2/3 | Language-agnostic, no UNK |
| SentencePiece (Unigram) | 32K | Probabilistic pruning | T5, mT5 | Better for multilingual |
| WordPiece | 30K-256K | Likelihood merging | BERT, Gemini, Gemma | Google ecosystem standard |
| Byte-level | 256 base | Direct UTF-8 | ByT5 | Zero OOV, but very long sequences |

### ⚡ Performance Impact

Tokenization efficiency directly affects inference cost. GPT-4's 100K vocabulary tokenizer compresses English text to ~1.3 tokens/word on average, while a 32K BPE tokenizer averages ~1.5 tokens/word. That 15% difference means 15% more compute for every prompt and response. For multilingual text, the gap widens dramatically — poorly tokenized languages can require 3-5× more tokens.

### ⚠️ Common Pitfalls

- **Tokenizer mismatch**: Using a different tokenizer at inference than at training silently corrupts every embedding lookup — always load the exact tokenizer the model was trained with.
- **Multilingual tax**: Tokenizers trained primarily on English can inflate non-English text by 3-5×, dramatically increasing cost and latency for multilingual applications.
- **Special token collisions**: Forgetting to handle `<|endoftext|>`, `<s>`, `</s>`, or `[PAD]` tokens correctly causes subtle generation bugs — always check the model's special token map.
- **Vocabulary size vs embedding memory**: Doubling vocab from 32K to 64K doubles the embedding table size (e.g., +256MB at d_model=4096 in float32) — this matters on memory-constrained devices.

### 🎯 Interview Tip

> *"BPE builds vocabulary bottom-up by iteratively merging the most frequent adjacent token pairs, while Unigram works top-down by starting with a large vocabulary and pruning tokens that least affect the likelihood. BPE is deterministic (one segmentation per string), while Unigram is probabilistic (can sample multiple valid segmentations) — this makes Unigram useful for data augmentation during training. Modern LLMs overwhelmingly use BPE variants because of their simplicity and good compression ratios."*